# 07 Cyclic permutation of the training-free 3-shot conditions

Applies the per-item cyclic permutation of notebook 06 to the base model with each of the three
3-shot conditions of the paper (answer-only, synthetic-format-only, rationale-CoT), so that
prompting and fine-tuning are compared under the same position control.

- The exemplars are built with the main pipeline's code, unchanged. The unrotated predictions
  (shift 0) are the existing `<condition>_ex<N>_<benchmark>.csv` files, so the gate [1-0] first
  regenerates part of shift 0 and requires every prediction to match.
- rationale-CoT generates up to 500 tokens, as in the main pipeline, but stops at the first `[[X]]`.
  The answer is extracted from the first `[[X]]`, so predictions are unchanged.
- Stage 1: ToMi, three conditions x three exemplar seeds. Stage 2: the other four benchmarks,
  three conditions x exemplar seed 0.
- [3] records the magnitude of the answer-letter logits on the items of notebook 04.
- [4] (optional, off by default) determines the actual weight dtype of the 16-bit control.
- [5] examines the rationale-CoT rotations in which no `[[X]]` marker was found.

Order: [0-1], restart, [0-2], [1-0], [3], [1], [2]; [4] and [5] as needed.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import time, torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB VRAM)')
t0 = time.time()
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s  ->  restart the runtime, then run from [0-2]')

In [ ]:
# -- [0-2] Mount Drive, set paths, helpers (same helpers as notebook 06, plus the few-shot setup) --
from google.colab import drive
drive.mount('/content/drive')

import os, re, json as _json, time, hashlib, platform, string
import pandas as pd, numpy as np, torch
import transformers, peft, unsloth
from unsloth import FastLanguageModel
transformers.logging.set_verbosity_error()   # hide max_new_tokens/max_length warnings (no effect on results)
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
REPRO_DIR   = f'{DRIVE_DIR}/repro_check'
CYC_DIR     = f'{DRIVE_DIR}/cyclic_itemlevel'

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
LETTERS = list(string.ascii_uppercase)

# adapter hashes; stop if a different adapter is found
EXPECTED_HASH = {42: '1de40b6b5c101d83', 43: '8f3570a519304ca5', 44: '96dbca35b3aa2b68',
                 45: '763ef681101c2872', 46: 'e1fad1151d5af492'}
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()[:16]
ADAPTERS = {s: f'{RESULTS_DIR}/adapter_seed{s}' for s in SEEDS}
for s, p in ADAPTERS.items():
    h = sha16(f'{p}/adapter_model.safetensors')
    assert h == EXPECTED_HASH[s], f'seed{s} adapter hash mismatch: {h}'
print('five adapter hashes confirmed')

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def build_user_prompt(story, question, options):
    lines = [f'{l}. {t}' for l, t in options]
    return f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines)

def parse_options(user_prompt):
    m = re.search(r'\[Candidate Answers\]\n(.+)', user_prompt, re.S)
    block = m.group(1) if m else user_prompt
    return [(l, t.strip()) for l, t in re.findall(r'^([A-E])\.\s*(.+?)$', block, re.M)]

def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        items = _json.load(f)
    for it in items:
        it['options'] = parse_options(it['user_prompt'])
        n = len(it['options'])
        assert [l for l, _ in it['options']] == LETTERS[:n] and it['answer'] in LETTERS[:n], it['item_id']
        # the rebuilt unrotated prompt must equal the original prompt character for character
        assert build_user_prompt(it['story'], it['question'], it['options']) == it['user_prompt'], it['item_id']
        it['n_options'] = n
    return items

MANIFESTS = {bm: load_manifest(bm) for bm in BENCHMARKS}
for bm in BENCHMARKS:
    print(bm, len(MANIFESTS[bm]), pd.Series([it['n_options'] for it in MANIFESTS[bm]]).value_counts().sort_index().to_dict())

def cyclic_shift(options, k):
    # moves the option at original position i to position (i+k) mod n; remap: original letter -> new letter
    n = len(options)
    texts = [t for _, t in options]
    shifted = texts[-k % n:] + texts[:-k % n] if k % n else texts
    new_options = [(LETTERS[i], shifted[i]) for i in range(n)]
    remap = {LETTERS[i]: LETTERS[(i + k) % n] for i in range(n)}
    return new_options, remap

# self-check: for every k the gold option text must sit at the new gold letter
for it in MANIFESTS['HiToM'][:20] + MANIFESTS['OpenToM'][:20] + MANIFESTS['ToMBench'][:50]:
    gold_text = dict(it['options'])[it['answer']]
    for k in range(it['n_options']):
        opts, remap = cyclic_shift(it['options'], k)
        assert dict(opts)[remap[it['answer']]] == gold_text
print('cyclic_shift self-check passed')

@torch.no_grad()
def infer_one(model, tokenizer, user_prompt, letter_ids, lid):
    inputs = tokenizer(to_text(tokenizer, user_prompt), return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id,
                         output_scores=True, return_dict_in_generate=True)
    gen_ids = out.sequences[0][inputs['input_ids'].shape[1]:]
    gen = tokenizer.decode(gen_ids, skip_special_tokens=True)
    margin = None
    for step, tok in enumerate(gen_ids.tolist()):
        if tok in letter_ids and step < len(out.scores):
            sc = out.scores[step][0].float().cpu()[lid]
            v, _ = torch.sort(sc, descending=True)
            margin = float(v[0] - v[1])
            break
    return extract_letter(gen), gen, margin

def letter_token_ids(tokenizer):
    ids = {}
    for L in 'ABCDE':
        t = tokenizer.encode(L, add_special_tokens=False)
        assert len(t) == 1, f'{L} is not a single token: {t}'
        ids[t[0]] = L
    return ids, torch.tensor(list(ids.keys()))

def out_csv(model_tag, bm):
    return f'{OUT_DIR}/cyclic_{model_tag}_{bm}.csv'

def run_cyclic(model_tag, model, tokenizer, k_start):
    # k_start=1: shift 0 is taken from existing results / k_start=0: all shifts from 0
    FastLanguageModel.for_inference(model)
    letter_ids, lid = letter_token_ids(tokenizer)
    from tqdm.auto import tqdm
    for bm in BENCHMARKS:
        path = out_csv(model_tag, bm)
        rows = pd.read_csv(path).to_dict('records') if os.path.exists(path) else []
        done = {(r['item_id'], int(r['shift_k'])) for r in rows}
        todo = [(it, k) for it in MANIFESTS[bm] for k in range(k_start, it['n_options']) if (it['item_id'], k) not in done]
        if not todo:
            print(f'[skip] {model_tag} {bm}: done ({len(rows)} rows)'); continue
        print(f'{model_tag} {bm}: {len(done)} rows done, {len(todo)} to go')
        t0 = time.time()
        for j, (it, k) in enumerate(tqdm(todo, desc=f'{model_tag}-{bm}')):
            opts, remap = cyclic_shift(it['options'], k)
            inv = {v: o for o, v in remap.items()}   # new letter -> original letter
            prompt = build_user_prompt(it['story'], it['question'], opts)
            pred, raw, margin = infer_one(model, tokenizer, prompt, letter_ids, lid)
            new_gold = remap[it['answer']]
            rows.append({'model': model_tag, 'benchmark': bm, 'item_id': it['item_id'], 'task': it.get('task'),
                         'n_options': it['n_options'], 'shift_k': k, 'gold_orig': it['answer'], 'new_gold': new_gold,
                         'pred': pred, 'pred_orig_letter': inv.get(pred), 'correct': int(pred == new_gold),
                         'letter_margin': margin, 'raw': raw})
            if (j + 1) % 500 == 0:
                pd.DataFrame(rows).to_csv(path, index=False)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'[saved] {model_tag} {bm}: {len(rows)} rows ({(time.time()-t0)/60:.1f} min)')

meta = {'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda, 'torch': torch.__version__,
        'transformers': transformers.__version__, 'peft': peft.__version__, 'unsloth': unsloth.__version__,
        'python': platform.python_version(), 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
os.makedirs(f'{DRIVE_DIR}/fewshot_cyclic', exist_ok=True)
with open(f'{DRIVE_DIR}/fewshot_cyclic/session_meta_{time.strftime("%Y%m%d_%H%M")}.json', 'w') as f:
    _json.dump(meta, f, indent=2)
print(meta)
# ===== few-shot settings (the part above is the same as [0-2] in notebook 06) =====
OUT_DIR = f'{DRIVE_DIR}/fewshot_cyclic'
os.makedirs(OUT_DIR, exist_ok=True)
import random
from transformers import StoppingCriteria, StoppingCriteriaList

# --- few-shot exemplar construction, character-for-character as in tombench_slm_qlora_complete_pipeline.ipynb ---
with open(f'{RESULTS_DIR}/split_train_full_v2.json', encoding='utf-8') as f:
    train_df = pd.DataFrame(_json.load(f))
assert len(train_df) == 1607

def pick_exemplars(exemplar_seed, k=3):
    idx = train_df.sample(n=k, random_state=100+exemplar_seed).index.tolist()
    return train_df.loc[idx]

def build_rationale_cot_fewshot(exemplar_seed):
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        reasoning = (f"Let's reason step by step: I track who has access to which information in the story, "
                     f"then check what the relevant character would infer given only what they observed. "
                     f"This points to option {r['answer']}.")
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"{reasoning} [[{r['answer']}]]"})
    return msgs

def build_answer_only_3shot(exemplar_seed):
    exemplars = pick_exemplars(exemplar_seed)
    msgs = []
    for _, r in exemplars.iterrows():
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"[[{r['answer']}]]"})
    return msgs

_SYNTH_TOPICS = [
    ('[Story]\nA red box and a blue box sit on a table.\n\n[Question]\nWhich box is red?\n\n[Candidate Answers]\nA. The red box\nB. The blue box', 'A'),
    ('[Story]\nThere are three cups: one has water, one has juice, one is empty.\n\n[Question]\nWhich cup is empty?\n\n[Candidate Answers]\nA. The water cup\nB. The juice cup\nC. The empty cup', 'C'),
    ('[Story]\nA number sequence: 2, 4, 6, 8.\n\n[Question]\nWhat comes next?\n\n[Candidate Answers]\nA. 9\nB. 10\nC. 12', 'B'),
    ('[Story]\nA clock shows 3:00. One hour passes.\n\n[Question]\nWhat time is it now?\n\n[Candidate Answers]\nA. 2:00\nB. 4:00\nC. 5:00', 'B'),
    ('[Story]\nA shelf has 5 books. 2 are removed.\n\n[Question]\nHow many books remain?\n\n[Candidate Answers]\nA. 2\nB. 3\nC. 5', 'B'),
]

def build_synthetic_format_only(exemplar_seed):
    rng = random.Random(200+exemplar_seed)
    picks = rng.sample(_SYNTH_TOPICS, 3)
    msgs = []
    for prompt, ans in picks:
        msgs.append({'role':'user','content': prompt})
        msgs.append({'role':'assistant','content': f'[[{ans}]]'})
    return msgs

CONDITIONS = {'answer_only_3shot': build_answer_only_3shot,
              'synthetic_format_only_3shot': build_synthetic_format_only,
              'rationale_cot_3shot': build_rationale_cot_fewshot}
MAX_NEW = {'answer_only_3shot': 12, 'synthetic_format_only_3shot': 12, 'rationale_cot_3shot': 500}

def to_text_fs(tokenizer, user_prompt, prefix):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}] + prefix + [{'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

class StopAfterAnswer(StoppingCriteria):
    # stop at the first [[X]]; extract_letter uses the first [[X]], so the prediction equals that of a 500-token generation
    def __init__(self, tokenizer, start): self.tok, self.start = tokenizer, start
    def __call__(self, input_ids, scores, **kw):
        hit = bool(ANS_RE.search(self.tok.decode(input_ids[0][self.start:], skip_special_tokens=True)))
        return torch.full((input_ids.shape[0],), hit, dtype=torch.bool, device=input_ids.device)

@torch.no_grad()
def infer_fs(model, tokenizer, user_prompt, prefix, max_new):
    inputs = tokenizer(to_text_fs(tokenizer, user_prompt, prefix), return_tensors='pt').to(model.device)
    start = inputs['input_ids'].shape[1]
    stop = StoppingCriteriaList([StopAfterAnswer(tokenizer, start)]) if max_new > 12 else None
    out = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id, stopping_criteria=stop)
    gen = tokenizer.decode(out[0][start:], skip_special_tokens=True)
    return extract_letter(gen), gen

def fs_csv(cond, ex, bm):
    return f'{OUT_DIR}/fscyc_{cond}_ex{ex}_{bm}.csv'

def run_fs_cyclic(model, tokenizer, cond, ex, bm):
    prefix = CONDITIONS[cond](ex)
    path = fs_csv(cond, ex, bm)
    rows = pd.read_csv(path).to_dict('records') if os.path.exists(path) else []
    done = {(r['item_id'], int(r['shift_k'])) for r in rows}
    todo = [(it, k) for it in MANIFESTS[bm] for k in range(1, it['n_options']) if (it['item_id'], k) not in done]
    if not todo:
        print(f'[skip] {cond} ex{ex} {bm}: done ({len(rows)} rows)'); return
    print(f'{cond} ex{ex} {bm}: {len(done)} rows done, {len(todo)} to go')
    from tqdm.auto import tqdm
    t0 = time.time()
    for j, (it, k) in enumerate(tqdm(todo, desc=f'{cond[:8]}-ex{ex}-{bm}')):
        opts, remap = cyclic_shift(it['options'], k)
        inv = {v: o for o, v in remap.items()}
        pred, raw = infer_fs(model, tokenizer, build_user_prompt(it['story'], it['question'], opts), prefix, MAX_NEW[cond])
        new_gold = remap[it['answer']]
        rows.append({'condition': cond, 'exemplar_seed': ex, 'benchmark': bm, 'item_id': it['item_id'],
                     'task': it.get('task'), 'n_options': it['n_options'], 'shift_k': k,
                     'gold_orig': it['answer'], 'new_gold': new_gold, 'pred': pred,
                     'pred_orig_letter': inv.get(pred), 'correct': int(pred == new_gold),
                     'marker_found': bool(ANS_RE.search(raw)), 'raw': raw})
        if (j + 1) % 300 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'[saved] {cond} ex{ex} {bm}: {len(rows)} rows ({(time.time()-t0)/60:.1f} min)')

STAGE1 = [(c, ex, 'ToMi') for c in CONDITIONS for ex in [0, 1, 2]]
STAGE2 = [(c, 0, bm) for c in CONDITIONS for bm in ['ToMBench', 'OpenToM', 'SocialIQa', 'HiToM']]
print('few-shot setup done. stage 1:', len(STAGE1), 'jobs, stage 2:', len(STAGE2), 'jobs')


---
## [1-0] Gate (about 10 min)

Regenerates shift 0 for the first ToMi items under each condition and exemplar seed and compares
with the existing results. Stop if there is any mismatch.

In [ ]:
# -- [1-0] Gate: check the rebuilt few-shot exemplars (regenerate part of shift 0 and compare with existing results) --
# shift 0 is taken from the existing few-shot results, so first confirm that the exemplars match the original ones.
base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(base_model)

N_CHECK = {'answer_only_3shot': 30, 'synthetic_format_only_3shot': 30, 'rationale_cot_3shot': 10}
gate_rows = []
for cond in CONDITIONS:
    for ex in [0, 1, 2]:
        ref = pd.read_csv(f'{RESULTS_DIR}/{cond}_ex{ex}_ToMi.csv').set_index('item_id')
        prefix = CONDITIONS[cond](ex)
        for it in MANIFESTS['ToMi'][:N_CHECK[cond]]:
            pred, raw = infer_fs(base_model, base_tok, it['user_prompt'], prefix, MAX_NEW[cond])
            gate_rows.append({'condition': cond, 'ex': ex, 'item_id': it['item_id'], 'pred': pred, 'pred_ref': ref.loc[it['item_id'], 'pred']})
g = pd.DataFrame(gate_rows); g['same'] = g['pred'] == g['pred_ref']
print(g.groupby(['condition', 'ex'])['same'].agg(['sum', 'count']))
g.to_csv(f'{OUT_DIR}/gate_k0_check.csv', index=False)
n_bad = int((~g['same']).sum())
print(f'\nmismatches {n_bad}/{len(g)}')
assert n_bad == 0, 'differs from the existing shift-0 results; the exemplars or the environment differ from the original run. Do not run [1].'
print('gate passed: exemplars and environment match the original run. Continue with [1].')


---
## [3] Magnitude of the answer-letter logits (about 5 min)

In [ ]:
# -- [3] Magnitude of the answer-letter logits --
# on the 107 items of the loading-path comparison (67 flipped + 40 stable, random_state=0), record the A-E logits at the position after '[['.
# a difference of 0.125 equals one bf16 step only for logits in [16, 32), so count how many items fall in that range.
from peft import PeftModel
REPRO_DIR = f'{DRIVE_DIR}/repro_check'
mm = pd.read_csv(f'{REPRO_DIR}/repro_margin_run1_vs_published.csv')
pick = pd.concat([mm[mm['flipped']], mm[~mm['flipped']].sample(40, random_state=0)])
by_id = {it['item_id']: it for bm in BENCHMARKS for it in MANIFESTS[bm]}

base_p, tok_p = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
m_peft = PeftModel.from_pretrained(base_p, ADAPTERS[42]); FastLanguageModel.for_inference(m_peft)
m_uns, tok_u = FastLanguageModel.from_pretrained(model_name=ADAPTERS[42], max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(m_uns)
letter_ids = [tok_u.encode(L, add_special_tokens=False)[0] for L in 'ABCDE']

@torch.no_grad()
def letter_logits(model, tok, up):
    ids = tok(to_text(tok, up) + '[[', return_tensors='pt').to(model.device)
    lg = model(**ids).logits[0, -1]
    return lg.dtype, lg[letter_ids].float().cpu().numpy()

def bf16_ulp(x):
    x = abs(float(x))
    return 2.0 ** (np.floor(np.log2(x)) - 7) if x > 0 else float('nan')

rows = []
from tqdm.auto import tqdm
for _, r in tqdm(pick.iterrows(), total=len(pick)):
    up = by_id[r['item_id']]['user_prompt']
    dt, lu = letter_logits(m_uns, tok_u, up); _, lp = letter_logits(m_peft, tok_p, up)
    top = int(np.argmax(lu))
    rows.append({'item_id': r['item_id'], 'benchmark': r['benchmark'], 'flipped': bool(r['flipped']), 'logit_dtype': str(dt),
                 **{f'uns_{L}': float(v) for L, v in zip('ABCDE', lu)}, **{f'peft_{L}': float(v) for L, v in zip('ABCDE', lp)},
                 'top_logit': float(lu[top]), 'second_logit': float(np.sort(lu)[-2]),
                 'bf16_ulp_at_top': bf16_ulp(lu[top]), 'max_abs_diff_letters': float(np.abs(lu - lp).max())})
lg = pd.DataFrame(rows)
lg['diff_in_ulp'] = lg['max_abs_diff_letters'] / lg['bf16_ulp_at_top']
lg.to_csv(f'{OUT_DIR}/answer_letter_logit_magnitude.csv', index=False)
print('logit dtype:', lg['logit_dtype'].unique())
print('\ntop-letter logit magnitude'); print(lg.groupby('flipped')['top_logit'].describe().round(3))
print('\nsecond-letter logit magnitude'); print(lg.groupby('flipped')['second_logit'].describe().round(3))
for lo, hi in [(8, 16), (16, 32), (32, 64)]:
    print(f'items with top logit in [{lo},{hi}): {int(lg["top_logit"].abs().between(lo, hi, inclusive="left").sum())}/{len(lg)} (one bf16 step = {2.0**(np.log2(lo)-7)})')
print('\nlargest answer-letter logit difference between the two paths, in bf16 steps at that magnitude'); print(lg.groupby('flipped')['diff_in_ulp'].describe().round(2))
del m_peft, base_p, m_uns; torch.cuda.empty_cache()


---
## [1] Few-shot cyclic permutation

In [ ]:
# -- [1] few-shot cyclic permutation (stage 1: all of ToMi -> stage 2: other benchmarks, ex0) --
if 'base_model' not in globals():
    base_model, base_tok = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(base_model)

RUN_STAGE2 = True   # False to run stage 1 only
for cond, ex, bm in STAGE1 + (STAGE2 if RUN_STAGE2 else []):
    run_fs_cyclic(base_model, base_tok, cond, ex, bm)
print('\n[1] done')


---
## [2] Summary (no GPU needed)

Also runs when only stage 1 is complete.

In [ ]:
# -- [2] Summary (no GPU needed) --
from google.colab import drive
drive.mount('/content/drive')
import os, pandas as pd, numpy as np
pd.set_option('display.width', 220)
DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
CYC_DIR     = f'{DRIVE_DIR}/cyclic_itemlevel'
OUT_DIR     = f'{DRIVE_DIR}/fewshot_cyclic'
CONDS = ['answer_only_3shot', 'synthetic_format_only_3shot', 'rationale_cot_3shot']
BMS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']

# base zero-shot rotations (shifts 0..K-1) come from notebook 06
base_cyc = pd.concat([pd.read_csv(f'{CYC_DIR}/cyclic_base_{bm}.csv') for bm in BMS], ignore_index=True)
info = base_cyc.drop_duplicates(['benchmark', 'item_id'])[['benchmark', 'item_id', 'task', 'n_options', 'gold_orig']]

frames = []
for cond in CONDS:
    for ex in [0, 1, 2]:
        for bm in BMS:
            p = f'{OUT_DIR}/fscyc_{cond}_ex{ex}_{bm}.csv'
            if not os.path.exists(p): continue
            d = pd.read_csv(p)
            k0 = pd.read_csv(f'{RESULTS_DIR}/{cond}_ex{ex}_{bm}.csv')[['item_id', 'answer', 'pred', 'correct']]
            k0 = k0.merge(info[info['benchmark'] == bm].drop(columns='benchmark'), on='item_id', how='left')
            assert k0['n_options'].notna().all(), f'{cond} ex{ex} {bm}: item_id not in the base rotations'
            k0 = pd.DataFrame({'condition': cond, 'exemplar_seed': ex, 'benchmark': bm, 'item_id': k0['item_id'],
                               'task': k0['task'], 'n_options': k0['n_options'].astype(int),
                               'shift_k': 0, 'gold_orig': k0['answer'], 'new_gold': k0['answer'], 'pred': k0['pred'],
                               'pred_orig_letter': k0['pred'], 'correct': k0['correct']})
            frames.append(pd.concat([k0, d[k0.columns]], ignore_index=True))
fs = pd.concat(frames, ignore_index=True)
fs['model'] = fs['condition'] + '_ex' + fs['exemplar_seed'].astype(str)
b = base_cyc.assign(model='base_zeroshot')[['model', 'benchmark', 'item_id', 'n_options', 'shift_k', 'pred', 'pred_orig_letter', 'correct']]
allr = pd.concat([b, fs[b.columns]], ignore_index=True)

cnt = allr.groupby(['model', 'benchmark', 'item_id']).agg(n=('shift_k', 'nunique'), K=('n_options', 'first'))
print('items with missing rotations:', int((cnt['n'] != cnt['K']).sum()))

def item_metrics(g):
    return pd.Series({'acc_orig': g.loc[g['shift_k'] == 0, 'correct'].mean(), 'lenient': g['correct'].mean(),
                      'strict': float(g['correct'].min() == 1), 'consistent': float(g['pred_orig_letter'].nunique() == 1)})
per_item = allr.groupby(['model', 'benchmark', 'item_id']).apply(item_metrics, include_groups=False).reset_index()
summ = per_item.groupby(['model', 'benchmark'])[['acc_orig', 'lenient', 'strict', 'consistent']].mean() * 100
base = summ.xs('base_zeroshot', level='model')
d = summ.drop('base_zeroshot', level='model').sub(base, level='benchmark').reset_index()
d['condition'] = d['model'].str.rsplit('_ex', n=1).str[0]
print('\n[few-shot - base zero-shot change, pp] by exemplar seed')
print(d.round(2).to_string(index=False))
agg = d.groupby(['condition', 'benchmark'])[['acc_orig', 'lenient', 'strict', 'consistent']].agg(['mean', 'std', 'count']).round(2)
print('\n[mean ± SD by condition (across exemplar seeds)]'); print(agg.to_string())
agg.to_csv(f'{OUT_DIR}/fewshot_cyclic_delta.csv')

ft = pd.read_csv(f'{CYC_DIR}/cyclic_delta_5seed.csv', header=[0, 1], index_col=0)
print('\n[comparison] fine-tuned five-seed change (notebook 06): lenient / strict / acc_orig')
print(ft[[('lenient', 'mean'), ('lenient', 'ci_lo'), ('lenient', 'ci_hi'), ('strict', 'mean'), ('acc_orig', 'mean')]].round(2).to_string())

print('\n[chosen-letter distribution (%), all rotations] uniform = 100/K')
rows = []
for (m, bm, K), g in allr.groupby(['model', 'benchmark', 'n_options']):
    dist = g['pred'].value_counts(normalize=True).reindex(list('ABCDE')[:K], fill_value=0) * 100
    rows.append({'model': m, 'benchmark': bm, 'K': K, 'max_minus_min': round(dist.max() - dist.min(), 1), **dist.round(1).to_dict()})
print(pd.DataFrame(rows).to_string(index=False))
cot = fs[fs['condition'] == 'rationale_cot_3shot']
if len(cot) and 'marker_found' in pd.read_csv(f'{OUT_DIR}/fscyc_rationale_cot_3shot_ex0_ToMi.csv').columns:
    mk = pd.concat([pd.read_csv(p) for p in [f'{OUT_DIR}/fscyc_rationale_cot_3shot_ex{e}_{bm}.csv' for e in [0, 1, 2] for bm in BMS] if os.path.exists(p)])
    print(f'\nrationale-CoT rotations without a [[X]] marker: {int((~mk["marker_found"].astype(bool)).sum())}/{len(mk)}')


---
## [4] (optional) Weight dtype of the 16-bit control

Set `RUN_DTYPE_CHECK = True` to run.

In [ ]:
# -- [4] (optional) Actual weight dtype of the 16-bit control, about 15 min --
# The control loads the non-quantized checkpoint with dtype=None, which gives fp16 on a T4 and bf16 on an L4; the GPU of the original sessions was not recorded.
# Load the same checkpoint in fp16 and in bf16, run ToMBench zero-shot on the 842 test items, and see which matches baseline_fp16_ToMBench.csv with 0 mismatches.
# If neither matches (e.g., kernel differences on another GPU), the dtype cannot be determined.
RUN_DTYPE_CHECK = False
if RUN_DTYPE_CHECK:
    ref = pd.read_csv(f'{RESULTS_DIR}/baseline_fp16_ToMBench.csv').set_index('item_id')
    letter_ids_d = None
    for name, dt in [('float16', torch.float16), ('bfloat16', torch.bfloat16)]:
        m, t = FastLanguageModel.from_pretrained(model_name='unsloth/Qwen2.5-3B-Instruct', max_seq_length=MAX_SEQ_LENGTH,
                                                 dtype=dt, load_in_4bit=False)
        FastLanguageModel.for_inference(m)
        lid_map, lid = letter_token_ids(t)
        preds = {}
        from tqdm.auto import tqdm
        for it in tqdm(MANIFESTS['ToMBench'], desc=name):
            preds[it['item_id']] = infer_one(m, t, it['user_prompt'], lid_map, lid)[0]
        s = pd.Series(preds)
        print(f'{name}: mismatches with the existing 16-bit control base {int((s != ref.loc[s.index, "pred"]).sum())}/{len(s)}')
        pd.DataFrame({'item_id': s.index, 'pred': s.values}).to_csv(f'{OUT_DIR}/dtype_check_{name}_ToMBench.csv', index=False)
        del m, t; torch.cuda.empty_cache()
else:
    print('skipped (RUN_DTYPE_CHECK=False)')


---
## [5] rationale-CoT rotations without a `[[X]]` marker (no GPU needed)

Counts these rotations by benchmark and exemplar seed and reports their length, whether the
fallback rule finds a letter, and whether the text looks like a repetition loop.

In [ ]:
# -- [5] rationale-CoT rotations without a [[X]] marker (no GPU needed, 1 min) --
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, glob

OUT = '/content/drive/MyDrive/ToMBench_Results/fewshot_cyclic'
files = glob.glob(OUT + '/fscyc_rationale_cot_3shot_ex*_*.csv')
d = pd.concat([pd.read_csv(p) for p in files], ignore_index=True)
d['raw'] = d['raw'].astype(str)
ok = d['marker_found'].astype(bool)
miss = d[~ok].copy()

print('no marker', len(miss), '/', len(d))
print(miss.groupby(['benchmark', 'exemplar_seed']).size().unstack(fill_value=0))

rate = (~ok).groupby(d['benchmark']).mean() * 100
print(rate.round(2))

miss['len'] = miss['raw'].str.len()
miss['has_letter'] = miss['raw'].str.contains(r'\b[A-E]\b')

def is_loop(t):
    return len(t) > 200 and t[-100:] in t[:-100]

miss['repeat'] = miss['raw'].apply(is_loop)
print(miss.groupby('benchmark')['len'].describe().round(0))
print('letter found %', round(miss['has_letter'].mean() * 100, 1))
print('possible repetition loops', int(miss['repeat'].sum()))

samples = miss.sample(min(3, len(miss)), random_state=0)
for t in samples['raw']:
    print('-----')
    print(t[-400:])
